In [1]:
import os
os.chdir('/Users/bamjy99/Desktop/experiment')
os.getcwd()

'/Users/bamjy99/Desktop/experiment'

In [2]:
!pip3 install pyreadstat
!pip3 install scikit-learn
!pip3 install openai
!pip3 install transformers
!pip3 install tqdm
!pip3 install matplotlib


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import pyreadstat

from tqdm import tqdm
from openai import OpenAI
from transformers import pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [5]:
#df1 is general, df2 is multiculture
df1, meta = pyreadstat.read_sav('/Users/bamjy99/Desktop/experiment/general.sav')
df2, meta = pyreadstat.read_sav('/Users/bamjy99/Desktop/experiment/multiculture.sav')

In [6]:
synthetic1 = pd.read_csv('./synthetic1_final_csv')
synthetic1 = synthetic1.drop('Unnamed: 0', axis=1)
synthetic2 = pd.read_csv('./synthetic2_final_csv')
synthetic2 = synthetic2.drop('Unnamed: 0', axis=1)

In [7]:
qf = pd.read_csv('./question.csv')

In [8]:
def replace_bracket_text(row):
    if pd.notnull(row['Specific']):
        return re.sub(r'\[.*?\]', str(row['Specific']), row['Question'])
    else:
        return row['Question']

qf['Question'] = qf.apply(replace_bracket_text, axis=1)
qf = qf.drop(columns=['Specific']) #Specific 열 삭제

In [9]:
max_options = 7

def extract_options(row):
    pattern = r'\(([a-g])\)([^()]+)'
    options = re.findall(pattern, row['Question'])
    question_only = re.split(r'\([a-h]\)', row['Question'])[0].strip()
    option_texts = [opt[1].strip() for opt in options]
    option_texts += [np.nan] * (max_options - len(option_texts))
    result = {'Question': question_only}
    for i, opt in enumerate(option_texts):
        result[f'option_{chr(ord("a")+i)}'] = opt
    return pd.Series(result)

# 적용
qf_new = qf.apply(extract_options, axis=1)
qf_new['Tag'] = qf['Tag']
qf_new['Number'] = qf['Number']

In [10]:
qf1 = qf_new[qf_new['Tag'] != 'Multiculture']
qf2 = qf_new

In [11]:
synthetic1 = synthetic1.merge(
    qf1[['Question', 'Number']],
    left_on='question',
    right_on='Question',
    how='left'
)

#synthetic1 = synthetic1.drop(['Question'], axis=1)

In [12]:
synthetic2 = synthetic2.merge(
    qf2[['Question', 'Number']],
    left_on='question',
    right_on='Question',
    how='left'
)
# synthetic2 = synthetic2.drop(['Question'], axis=1)

In [13]:
answer_dict = {'A': 1,
               'B': 2,
               'C': 3,
               'D': 4,
               'E': 5,
               'F': 6,
               'G': 7}

synthetic1['final'] = synthetic1['final'].map(answer_dict)
synthetic2['final'] = synthetic2['final'].map(answer_dict)

In [14]:
need = [
    'B2_1','B2_2','B2_3','B2_4','B2_6','B2_7','B2_8','B2_9',
    'B2_10','B2_12','B2_14','B2_11','B2_13','B2_15',
    'B12_3','B12_4','B15_1','B15_2','B15_3',
    'A1_1','A1_2','A1_3','A1_4','A1_5','A1_6','A1_7','A1_8','A1_9',
    'A1_10','A1_11','A1_12','A1_13','A1_14','A1_15',
    'B8_1','B8_2','B8_3',
    'B9_1','B9_2','B9_3','B9_4','B9_5','B9_6','B9_7','B9_8','B9_9','B9_10',
    'D1_1','D1_2','D1_3','D1_4','D1_5','D1_6','D1_7','D1_8','D1_9'
]

In [15]:
df1 = df1[need]
df2 = df2[need]
synthetic1 = synthetic1[synthetic1['Number'].isin(need)]
synthetic2 = synthetic2[synthetic2['Number'].isin(need)]

In [16]:
synthetic1_llama = synthetic1[synthetic1['model'] == 'meta-llama/Llama-4-Scout-17B-16E-Instruct']
synthetic2_llama = synthetic2[synthetic2['model'] == 'meta-llama/Llama-4-Scout-17B-16E-Instruct']
synthetic1_llama = synthetic1_llama.drop('model', axis=1)
synthetic2_llama = synthetic2_llama.drop('model', axis=1)

In [17]:
# 중복 제거
synthetic1_llama = synthetic1_llama.drop_duplicates(subset=['질문_ID', 'question'], keep='first')
synthetic2_llama = synthetic2_llama.drop_duplicates(subset=['질문_ID', 'question'], keep='first')

In [18]:
# 편의상 (질문_ID가 실상은 페르소나 넘버)
synthetic1_llama = synthetic1_llama.rename(columns={'질문_ID': 'ID'})
synthetic2_llama = synthetic2_llama.rename(columns={'질문_ID': 'ID'})

### 답변이 100% 같은 답인 변수 찾아내기

In [19]:
def find_numbers_with_high_agreement(df, threshold=1):
    grouped = df.groupby('Number')
    result = []
    for number, group in grouped:
        top_value_ratio = group['final'].value_counts(normalize=True).max()
        if top_value_ratio >= threshold:
            result.append(number)
    return result

In [20]:
synthetic1_llama_result = find_numbers_with_high_agreement(synthetic1_llama, 1)
synthetic2_llama_result = find_numbers_with_high_agreement(synthetic2_llama, 1)

In [21]:
print(synthetic1_llama_result)
print(synthetic2_llama_result)

['A1_13', 'A1_2', 'B12_3', 'B15_1', 'B9_10', 'B9_7', 'B9_8', 'B9_9', 'D1_1', 'D1_2', 'D1_7']
['A1_13', 'A1_14', 'A1_2', 'B12_3', 'B15_1', 'B15_3', 'B9_10', 'B9_8', 'D1_1', 'D1_2', 'D1_3', 'D1_7']


In [22]:
result = list(dict.fromkeys(
    synthetic1_llama_result +
    synthetic2_llama_result))

In [23]:
print(len(result))

14


### 100% 이상 같은 값인 질문만 뽑아내기

In [24]:
qf1_new = qf1[qf1['Number'].isin(result)].copy()
qf2_new = qf2[qf2['Number'].isin(result)].copy()

### 새로운 프롬프트, 새로운 데이터

In [25]:
model_list=[
    "meta-llama/Llama-4-Scout-17B-16E-Instruct"
]

In [26]:
client = OpenAI(
    api_key="zqZDTHZb2TK54nul5evuEECGERFf5nei",
    base_url="https://api.deepinfra.com/v1/openai",
)

### 일반 프롬프트

In [27]:
# 원래 프롬프트 with 약간의 수정
# 일반 청소년
results = []

# 질문 데이터프레임(qdf1) 반복
for q_idx, q_row in qf1_new.iterrows():
    question_text = q_row['Question']
    options = {col: q_row[col] for col in q_row.index if col.startswith('option_')}
    
    # 유효 옵션 추출
    valid_options = []
    for opt in ['a','b','c','d','e','f','g']:
        val = options.get(f'option_{opt}', np.nan)
        if not pd.isna(val):
            valid_options.append(f"({opt.upper()}) {val}")
    filtered_question = f"{question_text} " + " ".join(valid_options)
    
    # 페르소나 데이터 반복
    for p_idx, p_row in synthetic1_llama.iterrows():
        persona = str(p_row['persona'])
        id = p_row['ID']
        # 모델 리스트 반복
        for model in model_list:
            print(f"\n=== 질문 {q_idx} | 페르소나 {p_idx} | 모델: {model} ===")
            
            # 메시지 생성
            messages = [{
                "role": "user",
                "content": (
                    f"페르소나: '{persona}'\n"
                    f"질문: {filtered_question}\n"
                    f"보기: {valid_options}\n"
                    f"제약 조건: 주어진 보기 안에서 대답해줘. 한국어로만 답변하고 영어는 사용하지 마.\n"
                    f"명령: 명시된 페르소나 특성을 지닌 응답자의 관점에서 질문에 답해줘.\n"
                )
            }]
            
                # 초기 답변 생성
            chat_completion = client.chat.completions.create(
                    model=model,
                    messages=messages,
                )
            answer = chat_completion.choices[0].message.content.strip()
            print(f"초기 답변: {answer}")


                # 이유 답변 요청
            messages.append({"role": "assistant", "content": answer})
            messages.append({"role": "user", "content": "왜 그렇게 답변을 생각했어? 이유를 알려줘.\nLlm:"})
                
            reason_response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                )
            reason = reason_response.choices[0].message.content.strip()
            print(f"이유: {reason}")    



                # 결과 저장
            results.append({
                    '질문_ID': id,
                    'persona': persona,
                    'question': question_text,
                    'model': model,
                    'initial_answer': answer,
                    'reason': reason
                })
            
result_df1 = pd.DataFrame(results)
result_df1.to_csv('./weird_generating_general1.csv')


=== 질문 32 | 페르소나 12222 | 모델: meta-llama/Llama-4-Scout-17B-16E-Instruct ===
초기 답변: (B) 아니요


In [ ]:
# 원래 프롬프트 with 약간의 수정
# 일반 청소년
results = []

# 질문 데이터프레임(qdf2) 반복
for q_idx, q_row in qf2_new.iterrows():
    question_text = q_row['Question']
    options = {col: q_row[col] for col in q_row.index if col.startswith('option_')}
    
    # 유효 옵션 추출
    valid_options = []
    for opt in ['a','b','c','d','e','f','g']:
        val = options.get(f'option_{opt}', np.nan)
        if not pd.isna(val):
            valid_options.append(f"({opt.upper()}) {val}")
    filtered_question = f"{question_text} " + " ".join(valid_options)
    
    # 페르소나 데이터 반복
    for p_idx, p_row in synthetic2_llama.iterrows():
        persona = str(p_row['persona'])
        id = p_row['ID']
        # 모델 리스트 반복
        for model in model_list:
            print(f"\n=== 질문 {q_idx} | 페르소나 {p_idx} | 모델: {model} ===")
            
            # 메시지 생성
            messages = [{
                "role": "user",
                "content": (
                    f"페르소나: '{persona}'\n"
                    f"질문: {filtered_question}\n"
                    f"보기: {valid_options}\n"
                    f"제약 조건: 주어진 보기 안에서 대답해줘. 한국어로만 답변하고 영어는 사용하지 마.\n"
                    f"명령: 명시된 페르소나 특성을 지닌 응답자의 관점에서 질문에 답해줘.\n"
                )
            }]
            
                # 초기 답변 생성
            chat_completion = client.chat.completions.create(
                    model=model,
                    messages=messages,
                )
            answer = chat_completion.choices[0].message.content.strip()
            print(f"답변: {answer}")
                
                # 이유 답변 요청
            messages.append({"role": "assistant", "content": answer})
            messages.append({"role": "user", "content": "왜 그렇게 답변을 생각했어? 이유를 알려줘.\nLlm:"})
                
            reason_response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                )
            reason = reason_response.choices[0].message.content.strip()
            print(f"이유: {reason}")    



                # 결과 저장
            results.append({
                    '질문_ID': id,
                    'persona': persona,
                    'question': question_text,
                    'model': model,
                    'initial_answer': answer,
                    'reason': reason
                })
result_df2 = pd.DataFrame(results)  
result_df2.to_csv('./weird_generating_general2.csv')

### Chain of Thoughts 프롬프트

In [ ]:
# 변형 프롬프트 (CoT) with 추론 과정
# 일반 청소년
results = []

# 질문 데이터프레임(qdf1) 반복
for q_idx, q_row in qf1_new.iterrows():
    question_text = q_row['Question']
    options = {col: q_row[col] for col in q_row.index if col.startswith('option_')}
    
    # 유효 옵션 추출
    valid_options = []
    for opt in ['a','b','c','d','e','f','g']:
        val = options.get(f'option_{opt}', np.nan)
        if not pd.isna(val):
            valid_options.append(f"({opt.upper()}) {val}")
    filtered_question = f"{question_text} " + " ".join(valid_options)
    
    # 페르소나 데이터 반복
    for p_idx, p_row in synthetic1_llama.iterrows():
        persona = str(p_row['persona'])
        id = p_row['ID']
        # 모델 리스트 반복
        for model in model_list:
            print(f"\n=== 질문 {q_idx} | 페르소나 {p_idx} | 모델: {model} ===")
            
            # 메시지 생성
            messages = [{
                "role": "user",
                "content": (
                    f"페르소나: '{persona}'\n"
                    f"질문: {filtered_question}\n"
                    f"보기: {valid_options}\n"
                    f"제약 조건: 주어진 보기 안에서 대답해줘. 한국어로만 답변하고 영어는 사용하지 마. 단계 별로 나눠서 논리적으로 생각해봐. 논리 추론 과정도 단계별로 대답해줘.\n"
                    f"명령: 명시된 페르소나 특성을 지닌 응답자의 관점에서 질문에 답해줘.\n"
                    )
            }]
            
                # 초기 답변 생성
            chat_completion = client.chat.completions.create(
                    model=model,
                    messages=messages,
                )
            answer = chat_completion.choices[0].message.content.strip()
            print(f"답변: {answer}")
                
                # 이유 답변 요청
            messages.append({"role": "assistant", "content": answer})
            messages.append({"role": "user", "content": "왜 그렇게 답변을 생각했어? 이유를 알려줘.\nLlm:"})
                
            reason_response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                )
            reason = reason_response.choices[0].message.content.strip()
            print(f"이유: {reason}")    



                # 결과 저장
            results.append({
                    '질문_ID': id,
                    'persona': persona,
                    'question': question_text,
                    'model': model,
                    'initial_answer': answer,
                    'reason': reason
                })
            

result_df1_cot = pd.DataFrame(results)
result_df1_cot.to_csv('./weird_generating_cot1.csv')

In [ ]:
# 변형 프롬프트 (CoT) with 추론 과정
# 일반 청소년
results = []

# 질문 데이터프레임(qdf1) 반복
for q_idx, q_row in qf2_new.iterrows():
    question_text = q_row['Question']
    options = {col: q_row[col] for col in q_row.index if col.startswith('option_')}
    
    # 유효 옵션 추출
    valid_options = []
    for opt in ['a','b','c','d','e','f','g']:
        val = options.get(f'option_{opt}', np.nan)
        if not pd.isna(val):
            valid_options.append(f"({opt.upper()}) {val}")
    filtered_question = f"{question_text} " + " ".join(valid_options)
    
    # 페르소나 데이터 반복
    for p_idx, p_row in synthetic2_llama.iterrows():
        persona = str(p_row['persona'])
        id = p_row['ID']
        # 모델 리스트 반복
        for model in model_list:
            print(f"\n=== 질문 {q_idx} | 페르소나 {p_idx} | 모델: {model} ===")
            
            # 메시지 생성
            messages = [{
                "role": "user",
                "content": (
                    f"페르소나: '{persona}'\n"
                    f"질문: {filtered_question}\n"
                    f"보기: {valid_options}\n"
                    f"제약 조건: 주어진 보기 안에서 대답해줘. 한국어로만 답변하고 영어는 사용하지 마. 단계 별로 나눠서 논리적으로 생각해봐. 논리 추론 과정도 단계별로 대답해줘.\n"
                    f"명령: 명시된 페르소나 특성을 지닌 응답자의 관점에서 질문에 답해줘.\n"
                    )
            }]
            
                # 초기 답변 생성
            chat_completion = client.chat.completions.create(
                    model=model,
                    messages=messages,
                )
            answer = chat_completion.choices[0].message.content.strip()
            print(f"답변: {answer}")
                
                # 이유 답변 요청
            messages.append({"role": "assistant", "content": answer})
            messages.append({"role": "user", "content": "왜 그렇게 답변을 생각했어? 이유를 알려줘.\nLlm:"})
                
            reason_response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                )
            reason = reason_response.choices[0].message.content.strip()
            print(f"이유: {reason}")    



                # 결과 저장
            results.append({
                    '질문_ID': id,
                    'persona': persona,
                    'question': question_text,
                    'model': model,
                    'initial_answer': answer,
                    'reason': reason
                })
            

result_df2_cot = pd.DataFrame(results)  
result_df2_cot.to_csv('./weird_generating_cot2.csv')

### 답변 추출

In [ ]:
def extract_final_letter(text):
    if pd.isnull(text):
        return ''
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    if not lines:
        return ''
    last_line = lines[-1]
    m = re.match(r'^([A-Z])$', last_line)
    if m:
        return m.group(1)
    m = re.match(r'^\(?([A-Z])\)?$', last_line)
    if m:
        return m.group(1)
    all_caps = re.findall(r'\b([A-Z])\b', text)
    return all_caps[-1] if all_caps else ''

In [ ]:
result_df1['Final'] = result_df1['initial_answer'].apply(extract_final_letter)
result_df1_cot['Final'] = result_df1_cot['initial_answer'].apply(extract_final_letter)

result_df2['Final'] = result_df2['initial_answer'].apply(extract_final_letter)
result_df2_cot['Final'] = result_df2_cot['initial_answer'].apply(extract_final_letter)

In [ ]:
#답변 비교
print(result_df1['Final'].value_counts())
print(result_df1_cot['Final'].value_counts())
print("*" * 100)
print(result_df2['Final'].value_counts())
print(result_df2_cot['Final'].value_counts())